<a href="https://colab.research.google.com/github/miriamamin1213-ux/3-Fold/blob/main/HecktorGPT2Unfrozen2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# HECKTOR GPT-2 Last 2 Blocks Unfrozen
# HPV Status Prediction | Compact Single-Cell Notebook
# Stratified 3-Fold Cross-Validation (Entire Dataset)
# ============================================================

# 1. IMPORT LIBRARIES, SET SEEDS AND SELECT DEVICE
import os, random, gdown
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2Model
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report, balanced_accuracy_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)

SEED = 42

def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# 2. LOAD HECKTOR EXCEL DATA
DATA_FILE = "HPV2025.xlsx"

if not os.path.exists(DATA_FILE):
    downloaded = gdown.download(
        id="1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv",
        output=DATA_FILE, quiet=False
    )
    if downloaded is None or not os.path.exists(DATA_FILE):
        raise FileNotFoundError("Could not download HPV2025.xlsx.")

df = pd.read_excel(DATA_FILE)

TARGET = "HPV Status"

FEATURES = [
    "Age", "Gender", "Tobacco Consumption", "Alcohol Consumption",
    "Performance Status", "T-stage", "N-stage", "M-stage"
]

missing = [c for c in [TARGET] + FEATURES if c not in df.columns]

if missing:
    raise KeyError(f"Required Excel columns missing: {missing}")


# 3. ROBUSTLY NORMALISE TARGET AND SELECT FEATURES
def normalize_target(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    if s in {"0", "0.0", "negative", "neg", "false", "no"}:
        return 0

    if s in {"1", "1.0", "positive", "pos", "true", "yes"}:
        return 1

    try:
        n = float(s)
        if n in (0, 1):
            return int(n)
    except (ValueError, TypeError):
        pass

    return np.nan


df["HPV_Label"] = df[TARGET].apply(normalize_target)

invalid_count = int(df["HPV_Label"].isna().sum())

df = df.dropna(subset=["HPV_Label"]).copy()
df["HPV_Label"] = df["HPV_Label"].astype(int)
df = df[FEATURES + ["HPV_Label"]].copy()

if df.empty:
    raise ValueError("No recognised HPV labels found; check the target column.")

counts = df["HPV_Label"].value_counts().reindex([0, 1], fill_value=0)

if (counts < 3).any():
    raise ValueError(
        f"Insufficient examples for 3-fold stratified CV: {counts.to_dict()}"
    )

print(
    "Dataset:", df.shape,
    "| Invalid/missing target rows dropped:", invalid_count
)
print("HPV classes (0=negative, 1=positive):\n", counts)


# 4. STRATIFIED 3-FOLD CROSS-VALIDATION USING THE ENTIRE DATASET
N_SPLITS = 3

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

X = df[FEATURES].reset_index(drop=True)
y = df["HPV_Label"].to_numpy(dtype=int)

print(
    f"\nUsing the entire labelled dataset in "
    f"{N_SPLITS}-fold stratified CV."
)
print(
    "Each fold is used once for validation; "
    "the other two folds are used for training."
)


# 5. IMPUTE MISSING FEATURES USING EACH FOLD'S TRAINING DATA ONLY
def impute_fold_features(train_df, val_df):
    train_df = train_df.copy()
    val_df = val_df.copy()

    fill_values = {}

    for col in FEATURES:

        if pd.api.types.is_numeric_dtype(train_df[col]):
            value = train_df[col].median()

            if pd.isna(value):
                value = 0

        else:
            modes = train_df[col].dropna().astype(str).mode()
            value = modes.iloc[0] if not modes.empty else "Unknown"

        fill_values[col] = value

        for split in (train_df, val_df):
            split[col] = split[col].fillna(value)

            if split[col].dtype == object:
                split[col] = split[col].astype(str).replace(
                    {"": "Unknown", "nan": "Unknown", "None": "Unknown"}
                )

    return train_df, val_df, fill_values


# 6. CONVERT CLINICAL FEATURES INTO TEXT PROMPTS
def display_value(x):
    if pd.isna(x):
        return "Unknown"

    if isinstance(x, (float, np.floating)) and float(x).is_integer():
        return str(int(x))

    return str(x).strip()


def format_stage(x, prefix):
    value = display_value(x)

    if value == "Unknown" or value.upper().startswith(prefix):
        return value

    try:
        n = float(value)

        if n.is_integer():
            return f"{prefix}{int(n)}"

    except ValueError:
        pass

    return f"{prefix}{value}"


def patient_to_text(row):
    labels = {
        "Age": "Age",
        "Gender": "Gender",
        "Tobacco Consumption": "Tobacco consumption",
        "Alcohol Consumption": "Alcohol consumption",
        "Performance Status": "Performance status",
        "T-stage": "Tumour T stage",
        "N-stage": "Nodal N stage",
        "M-stage": "Metastatic M stage"
    }

    parts = [
        "Q: Predict this patient's HPV status from the following HECKTOR clinical information."
    ]

    for col in FEATURES:
        value = display_value(row[col])

        if col in ("T-stage", "N-stage", "M-stage"):
            value = format_stage(row[col], col[0])

        parts.append(f"{labels[col]}: {value}.")

    parts.append("Is this patient HPV positive? Answer Yes or No.\nA:")

    return " ".join(parts)


# 7. LOAD TOKENIZER AND SET TRAINING PARAMETERS
MODEL_NAME = "openai-community/gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

BATCH_SIZE, EPOCHS = 64, 500

LEARNING_RATE, WEIGHT_DECAY, DROPOUT = 1e-3, 1e-4, 0.1

print("Batch size:", BATCH_SIZE, "| Epochs:", EPOCHS)


# 8. BUILD DATASETS AND DATA LOADERS
class HPVTextDataset(Dataset):

    def __init__(self, texts, labels, max_length):
        self.items = []

        for text, label in zip(texts, labels):
            enc = tokenizer(
                text,
                max_length=max_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )

            self.items.append((
                enc["input_ids"].squeeze(0),
                enc["attention_mask"].squeeze(0),
                int(label)
            ))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        ids, mask, label = self.items[i]

        return {
            "input_ids": ids,
            "attention_mask": mask,
            "label": torch.tensor(label, dtype=torch.long)
        }


# 9. INITIALISE GPT-2 CLASSIFIER AND CLASS-WEIGHTED LOSS
class HPVNetGPT2Text(nn.Module):

    def __init__(self):
        super().__init__()

        self.gpt2 = GPT2Model.from_pretrained(MODEL_NAME)
        self.gpt2.config.use_cache = False
        self.gpt2.config.pad_token_id = tokenizer.pad_token_id

        h = self.gpt2.config.hidden_size

        self.classifier = nn.Sequential(
            nn.LayerNorm(h),
            nn.Dropout(DROPOUT),
            nn.Linear(h, 128),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(128, 2)
        )

    def forward(self, input_ids, attention_mask):

        out = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        ).last_hidden_state

        positions = attention_mask.sum(dim=1) - 1

        pooled = out[
            torch.arange(out.size(0), device=out.device),
            positions
        ]

        return self.classifier(pooled)


def configure_model():

    model = HPVNetGPT2Text().to(device)

    # Freeze all GPT-2 parameters
    for param in model.gpt2.parameters():
        param.requires_grad = False

    # Unfreeze the final 2 GPT-2 blocks
    for block in model.gpt2.h[-2:]:
        for param in block.parameters():
            param.requires_grad = True

    # Keep the classifier head trainable
    for param in model.classifier.parameters():
        param.requires_grad = True

    return model


# 10. VALIDATION SCORING AND THRESHOLD SELECTION
def get_scores(model, loader):

    model.eval()

    scores, labels = [], []

    with torch.no_grad():
        for batch in loader:

            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)

            logits = model(ids, mask)

            scores.extend(
                torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            )

            labels.extend(batch["label"].numpy())

    return np.asarray(scores), np.asarray(labels, dtype=int)


def evaluate_loss(model, loader, class_weights):

    model.eval()

    total, n_total = 0.0, 0

    with torch.no_grad():
        for b in loader:

            ids = b["input_ids"].to(device)
            mask = b["attention_mask"].to(device)
            labels = b["label"].to(device)

            losses = nn.functional.cross_entropy(
                model(ids, mask),
                labels,
                reduction="none"
            )

            loss = (losses * class_weights[labels]).mean()

            n = len(labels)
            total += loss.item() * n
            n_total += n

    return total / max(n_total, 1)


def choose_threshold(scores, labels):

    thresholds = np.unique(np.r_[0.0, scores, 1.0])

    values = [
        (float(t), balanced_accuracy_score(labels, scores >= t))
        for t in thresholds
    ]

    return max(
        values,
        key=lambda x: (x[1], -abs(x[0] - 0.5))
    )


# 11. RUN 3-FOLD CROSS-VALIDATION AND SAVE BEST VALIDATION CHECKPOINTS
OUT_DIR = Path("/content/drive/MyDrive/HECKTOR_GPT2_Last2_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_names = [
    "Validation_Loss",
    "Validation_ROC_AUC",
    "Validation_PR_AUC",
    "Validation_Balanced_Accuracy"
]

fold_results = []

oof_predictions = {
    name: {"labels": [], "scores": [], "predictions": []}
    for name in checkpoint_names
}

all_fold_histories = []
all_fold_checkpoint_info = []


for fold, (train_idx, val_idx) in enumerate(
    skf.split(X, y), start=1
):

    print(f"\n{'=' * 22} FOLD {fold}/{N_SPLITS} {'=' * 22}")

    seed_everything(SEED + fold)

    train_df = X.iloc[train_idx].copy()
    val_df = X.iloc[val_idx].copy()

    y_train = y[train_idx]
    y_val = y[val_idx]

    # Fit imputation using this fold's training data only
    train_df, val_df, fill_values = impute_fold_features(
        train_df, val_df
    )

    # Convert clinical information into text prompts
    train_texts = [
        patient_to_text(r) for _, r in train_df.iterrows()
    ]

    val_texts = [
        patient_to_text(r) for _, r in val_df.iterrows()
    ]

    # Determine maximum sequence length from training prompts
    lengths = [
        len(tokenizer.encode(t, add_special_tokens=False))
        for t in train_texts
    ]

    MAX_LENGTH = min(
        1024,
        max(32, int(np.ceil(max(lengths) / 8) * 8))
    )

    print("Train:", len(y_train), "| Validation:", len(y_val))
    print("Train class counts:", np.bincount(y_train, minlength=2))
    print("Validation class counts:", np.bincount(y_val, minlength=2))
    print(
        "Max prompt length:", max(lengths),
        "| MAX_LENGTH:", MAX_LENGTH
    )

    train_ds = HPVTextDataset(train_texts, y_train, MAX_LENGTH)
    val_ds = HPVTextDataset(val_texts, y_val, MAX_LENGTH)

    loader_args = {
        "batch_size": BATCH_SIZE,
        "num_workers": 0,
        "pin_memory": torch.cuda.is_available()
    }

    generator = torch.Generator().manual_seed(SEED + fold)

    train_loader = DataLoader(
        train_ds,
        shuffle=True,
        generator=generator,
        **loader_args
    )

    val_loader = DataLoader(
        val_ds,
        shuffle=False,
        **loader_args
    )

    # A fresh GPT-2 model is initialised for every fold
    model = configure_model()

    # Calculate class weights using this fold's training data
    class_counts = np.bincount(y_train, minlength=2)

    class_weights = torch.tensor(
        len(y_train) / (2 * np.maximum(class_counts, 1)),
        dtype=torch.float32,
        device=device
    )

    def weighted_loss(logits, labels):

        losses = nn.functional.cross_entropy(
            logits,
            labels,
            reduction="none"
        )

        return (losses * class_weights[labels]).mean()

    # Optimise only trainable parameters
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    # Verify trainable parameters
    trainable_params = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )

    total_params = sum(
        p.numel() for p in model.parameters()
    )

    print(
        f"Trainable parameters: {trainable_params:,} / {total_params:,}"
    )

    print(
        "Trainable GPT-2 blocks: final 2; "
        "classifier head: trainable"
    )

    # Save checkpoints separately for each fold
    fold_dir = OUT_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    paths = {
        "Validation_Loss":
            fold_dir / "best_model_validation_loss.pth",

        "Validation_ROC_AUC":
            fold_dir / "best_model_validation_roc_auc.pth",

        "Validation_PR_AUC":
            fold_dir / "best_model_validation_pr_auc.pth",

        "Validation_Balanced_Accuracy":
            fold_dir / "best_model_validation_balanced_accuracy.pth"
    }

    best = {
        "Validation_Loss": float("inf"),
        "Validation_ROC_AUC": -float("inf"),
        "Validation_PR_AUC": -float("inf"),
        "Validation_Balanced_Accuracy": -float("inf")
    }

    best_info, history = {}, []

    # Training loop
    for epoch in range(1, EPOCHS + 1):

        model.train()

        loss_sum, n_sum = 0.0, 0

        for b in train_loader:

            ids = b["input_ids"].to(device)
            mask = b["attention_mask"].to(device)
            labels = b["label"].to(device)

            optimizer.zero_grad(set_to_none=True)

            loss = weighted_loss(model(ids, mask), labels)

            loss.backward()
            optimizer.step()

            n = len(labels)
            loss_sum += loss.item() * n
            n_sum += n

        train_loss = loss_sum / max(n_sum, 1)

        val_loss = evaluate_loss(
            model, val_loader, class_weights
        )

        val_scores, val_labels = get_scores(
            model, val_loader
        )

        val_auc = roc_auc_score(val_labels, val_scores)
        val_ap = average_precision_score(val_labels, val_scores)

        threshold, val_ba = choose_threshold(
            val_scores, val_labels
        )

        metrics = {
            "Validation_Loss": val_loss,
            "Validation_ROC_AUC": val_auc,
            "Validation_PR_AUC": val_ap,
            "Validation_Balanced_Accuracy": val_ba
        }

        history.append({
            "Fold": fold,
            "Epoch": epoch,
            "Train_Loss": train_loss,
            **metrics,
            "Validation_Threshold": threshold
        })

        # Save best checkpoint according to each validation metric
        for name, value in metrics.items():

            improved = (
                value < best[name]
                if name == "Validation_Loss"
                else value > best[name]
            )

            if improved:

                best[name] = float(value)

                best_info[name] = {
                    "epoch": epoch,
                    "value": float(value),
                    "threshold": threshold
                }

                torch.save(model.state_dict(), paths[name])

        if epoch % 50 == 0:

            print(
                f"Fold {fold} | Epoch {epoch}/{EPOCHS} | "
                f"Train Loss={train_loss:.4f} | "
                f"Val Loss={val_loss:.4f} | "
                f"ROC-AUC={val_auc:.4f} | "
                f"PR-AUC={val_ap:.4f} | "
                f"BA={val_ba:.4f} | "
                f"Threshold={threshold:.4f}"
            )

    # Save training history for this fold
    history_df = pd.DataFrame(history)

    history_df.to_csv(
        fold_dir / "training_history.csv",
        index=False
    )

    all_fold_histories.append(history_df)

    # Save best checkpoint information for this fold
    pd.DataFrame([
        {"Fold": fold, "Checkpoint": k, **v, "Path": str(paths[k])}
        for k, v in best_info.items()
    ]).to_csv(
        fold_dir / "best_validation_checkpoints.csv",
        index=False
    )

    all_fold_checkpoint_info.extend([
        {"Fold": fold, "Checkpoint": k, **v, "Path": str(paths[k])}
        for k, v in best_info.items()
    ])

    # Evaluate each selected checkpoint on this fold's validation data.
    # Each patient is validated only in the fold where they were held out.
    for name, path in paths.items():

        model.load_state_dict(
            torch.load(path, map_location=device)
        )

        scores, labels = get_scores(model, val_loader)

        threshold = best_info[name]["threshold"]
        preds = (scores >= threshold).astype(int)

        auc = roc_auc_score(labels, scores)
        ap = average_precision_score(labels, scores)
        ba = balanced_accuracy_score(labels, preds)
        f1 = f1_score(labels, preds, zero_division=0)

        cm = confusion_matrix(
            labels, preds, labels=[0, 1]
        )

        print(
            f"\nFold {fold} | {name} | "
            f"Best epoch {best_info[name]['epoch']} | "
            f"Threshold {threshold:.4f}"
        )

        print(classification_report(
            labels,
            preds,
            labels=[0, 1],
            target_names=["HPV Negative", "HPV Positive"],
            digits=4,
            zero_division=0
        ))

        print(
            "Confusion matrix [[TN, FP], [FN, TP]]:\n",
            cm
        )

        print(
            f"Validation ROC-AUC={auc:.4f} | "
            f"PR-AUC={ap:.4f} | "
            f"BA={ba:.4f} | F1={f1:.4f}"
        )

        fold_results.append({
            "Dataset": "HECKTOR",
            "Model": "GPT-2 Classifier",
            "Fold": fold,
            "Checkpoint_Selected_By": name,
            "Best_Epoch": best_info[name]["epoch"],
            "Validation_Threshold": threshold,
            "Validation_ROC_AUC": auc,
            "Validation_PR_AUC": ap,
            "Validation_Balanced_Accuracy": ba,
            "Validation_F1": f1,
            "TN": cm[0, 0],
            "FP": cm[0, 1],
            "FN": cm[1, 0],
            "TP": cm[1, 1]
        })

        oof_predictions[name]["labels"].extend(
            labels.tolist()
        )

        oof_predictions[name]["scores"].extend(
            scores.tolist()
        )

        oof_predictions[name]["predictions"].extend(
            preds.tolist()
        )

    # Clear fold-specific objects before starting the next fold
    del model, optimizer, train_loader, val_loader, train_ds, val_ds

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# 12. SAVE FOLD RESULTS AND MEAN ± STANDARD DEVIATION
fold_results_df = pd.DataFrame(fold_results)

fold_results_df.to_csv(
    OUT_DIR / "validation_metrics_by_fold.csv",
    index=False
)

metric_columns = [
    "Validation_Balanced_Accuracy",
    "Validation_ROC_AUC",
    "Validation_PR_AUC",
    "Validation_F1"
]

summary_rows = []

for checkpoint, group in fold_results_df.groupby(
    "Checkpoint_Selected_By"
):

    row = {
        "Dataset": "HECKTOR",
        "Model": "GPT-2 Classifier",
        "Checkpoint_Selected_By": checkpoint,
        "Number_of_Folds": len(group)
    }

    for metric in metric_columns:

        row[f"{metric}_Mean"] = group[metric].mean()
        row[f"{metric}_SD"] = group[metric].std(ddof=1)

    summary_rows.append(row)


summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    OUT_DIR / "cross_validation_mean_sd.csv",
    index=False
)

print("\n=== 3-FOLD CROSS-VALIDATION SUMMARY (MEAN ± SD) ===")
print(summary_df.to_string(index=False))

pd.DataFrame(all_fold_checkpoint_info).to_csv(
    OUT_DIR / "best_validation_checkpoints_all_folds.csv",
    index=False
)

pd.concat(
    all_fold_histories,
    ignore_index=True
).to_csv(
    OUT_DIR / "training_history_all_folds.csv",
    index=False
)


# 13. SAVE OUT-OF-FOLD ROC AND PRECISION-RECALL CURVES
for name, values in oof_predictions.items():

    labels = np.asarray(values["labels"], dtype=int)
    scores = np.asarray(values["scores"], dtype=float)
    preds = np.asarray(values["predictions"], dtype=int)

    safe = name.lower()

    # Each labelled patient contributes one out-of-fold prediction
    # per checkpoint-selection criterion.
    pd.DataFrame({
        "True_Label": labels,
        "HPV_Positive_Score": scores,
        "Predicted_Label": preds
    }).to_csv(
        OUT_DIR / f"oof_predictions_{safe}.csv",
        index=False
    )

    fpr, tpr, roc_thresholds = roc_curve(labels, scores)

    precision, recall, _ = precision_recall_curve(
        labels, scores
    )

    auc = roc_auc_score(labels, scores)
    ap = average_precision_score(labels, scores)

    pd.DataFrame({
        "FPR": fpr,
        "TPR": tpr,
        "Threshold": roc_thresholds
    }).to_csv(
        OUT_DIR / f"roc_coordinates_{safe}.csv",
        index=False
    )

    pd.DataFrame({
        "Precision": precision,
        "Recall": recall
    }).to_csv(
        OUT_DIR / f"pr_coordinates_{safe}.csv",
        index=False
    )

    # ROC curve
    plt.figure(figsize=(6, 5))

    plt.plot(
        fpr, tpr,
        label=f"AUC={auc:.3f}"
    )

    plt.plot([0, 1], [0, 1], "--")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")

    plt.title(f"HECKTOR Out-of-Fold ROC — {name}")

    plt.legend()
    plt.tight_layout()

    plt.savefig(
        OUT_DIR / f"roc_{safe}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    # Precision-recall curve
    plt.figure(figsize=(6, 5))

    plt.plot(
        recall, precision,
        label=f"AP={ap:.3f}"
    )

    plt.xlabel("Recall")
    plt.ylabel("Precision")

    plt.title(f"HECKTOR Out-of-Fold Precision-Recall — {name}")

    plt.legend()
    plt.tight_layout()

    plt.savefig(
        OUT_DIR / f"pr_{safe}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# 14. SAVE SETTINGS AND COMPARISON RESULTS
settings = {
    "Dataset": "HECKTOR",
    "Model": MODEL_NAME,
    "Target": "HPV Status",
    "Features": ", ".join(FEATURES),
    "Number_of_Features": len(FEATURES),

    "Split": (
        "Stratified 3-fold cross-validation using the entire "
        "labelled dataset; no separate holdout test set"
    ),

    "Number_of_Folds": N_SPLITS,
    "Seed": SEED,
    "Epochs": EPOCHS,
    "Batch_Size": BATCH_SIZE,
    "Learning_Rate": LEARNING_RATE,
    "Weight_Decay": WEIGHT_DECAY,
    "Dropout": DROPOUT,
    "Optimizer": "AdamW",
    "Class_Weighted_Loss": True,

    "GPT2_Backbone_Frozen": (
        "Partially frozen; final 2 blocks trainable"
    ),

    "GPT2_Trainable_Blocks": 2,

    "Checkpoint_Selection": (
        "Validation loss, ROC-AUC, PR-AUC, balanced accuracy"
    ),

    "Threshold_Selection": (
        "Maximum validation balanced accuracy"
    ),

    "Imputation": (
        "Fit on each fold's training data only"
    )
}

pd.DataFrame([settings]).to_csv(
    OUT_DIR / "settings.csv",
    index=False
)

comparison_file = Path("model_roc_pr_auc_comparison.csv")

comparison = summary_df[[
    "Dataset",
    "Model",
    "Checkpoint_Selected_By",
    "Validation_Balanced_Accuracy_Mean",
    "Validation_Balanced_Accuracy_SD",
    "Validation_ROC_AUC_Mean",
    "Validation_ROC_AUC_SD",
    "Validation_PR_AUC_Mean",
    "Validation_PR_AUC_SD"
]].copy()

# Keep these CV results separate from older holdout-test results
cv_comparison_file = Path("model_3fold_cv_comparison.csv")

if cv_comparison_file.exists():

    old = pd.read_csv(cv_comparison_file)

    combined = pd.concat(
        [old, comparison],
        ignore_index=True,
        sort=False
    )

    combined = combined.drop_duplicates(
        subset=[
            "Dataset",
            "Model",
            "Checkpoint_Selected_By"
        ],
        keep="last"
    )

else:
    combined = comparison

combined.to_csv(
    cv_comparison_file,
    index=False
)

print("\nAll outputs saved to:", OUT_DIR.resolve())

print(
    "Fold-level metrics:",
    (OUT_DIR / "validation_metrics_by_fold.csv").resolve()
)

print(
    "Mean ± SD summary:",
    (OUT_DIR / "cross_validation_mean_sd.csv").resolve()
)

print(
    "3-fold comparison CSV:",
    cv_comparison_file.resolve()
)

Device: cuda


Downloading...
From: https://drive.google.com/uc?id=1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv
To: /content/HPV2025.xlsx
100%|██████████| 66.0k/66.0k [00:00<00:00, 17.3MB/s]


Dataset: (588, 9) | Invalid/missing target rows dropped: 138
HPV classes (0=negative, 1=positive):
 HPV_Label
0     58
1    530
Name: count, dtype: int64

Using the entire labelled dataset in 3-fold stratified CV.
Each fold is used once for validation; the other two folds are used for training.


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

... (9 outputs left out)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Trainable parameters: 14,275,970 / 124,540,034
Trainable GPT-2 blocks: final 2; classifier head: trainable
Fold 2 | Epoch 50/500 | Train Loss=0.5905 | Val Loss=0.5660 | ROC-AUC=0.7974 | PR-AUC=0.9705 | BA=0.7780 | Threshold=0.2431
Fold 2 | Epoch 100/500 | Train Loss=0.5450 | Val Loss=0.5203 | ROC-AUC=0.8211 | PR-AUC=0.9765 | BA=0.7591 | Threshold=0.5749
Fold 2 | Epoch 150/500 | Train Loss=0.5303 | Val Loss=0.5007 | ROC-AUC=0.8262 | PR-AUC=0.9778 | BA=0.7696 | Threshold=0.2603
Fold 2 | Epoch 200/500 | Train Loss=0.4877 | Val Loss=0.5211 | ROC-AUC=0.8479 | PR-AUC=0.9808 | BA=0.7941 | Threshold=0.3668
Fold 2 | Epoch 250/500 | Train Loss=0.4646 | Val Loss=0.7708 | ROC-AUC=0.7780 | PR-AUC=0.9600 | BA=0.7235 | Threshold=0.5241
Fold 2 | Epoch 300/500 | Train Loss=0.4548 | Val Loss=0.6450 | ROC-AUC=0.8155 | PR-AUC=0.9761 | BA=0.7480 | Threshold=0.3188
Fold 2 | Epoch 350/500 | Train Loss=0.5165 | Val Loss=0.5306 | ROC-AUC=0.8170 | PR-AUC=0.9765 | BA=0.7544 | Threshold=0.6325
Fold 2 | Epoch 400/

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Trainable parameters: 14,275,970 / 124,540,034
Trainable GPT-2 blocks: final 2; classifier head: trainable
Fold 3 | Epoch 50/500 | Train Loss=0.6690 | Val Loss=0.6648 | ROC-AUC=0.7105 | PR-AUC=0.9565 | BA=0.6807 | Threshold=0.4943
Fold 3 | Epoch 100/500 | Train Loss=0.6732 | Val Loss=0.6653 | ROC-AUC=0.6821 | PR-AUC=0.9307 | BA=0.6898 | Threshold=0.4208
Fold 3 | Epoch 150/500 | Train Loss=0.6687 | Val Loss=0.6760 | ROC-AUC=0.6071 | PR-AUC=0.8994 | BA=0.6750 | Threshold=0.5793
Fold 3 | Epoch 200/500 | Train Loss=0.6761 | Val Loss=0.6810 | ROC-AUC=0.6526 | PR-AUC=0.9250 | BA=0.6875 | Threshold=0.5900
Fold 3 | Epoch 250/500 | Train Loss=0.6563 | Val Loss=0.6709 | ROC-AUC=0.6452 | PR-AUC=0.9267 | BA=0.6483 | Threshold=0.5200
Fold 3 | Epoch 300/500 | Train Loss=0.6369 | Val Loss=0.6916 | ROC-AUC=0.6301 | PR-AUC=0.9196 | BA=0.6233 | Threshold=0.5977
Fold 3 | Epoch 350/500 | Train Loss=0.6645 | Val Loss=0.7540 | ROC-AUC=0.5659 | PR-AUC=0.9003 | BA=0.6273 | Threshold=0.2411
Fold 3 | Epoch 400/